# 03 — PCA Baseline, Experiment 1 (Phase 3–4)

Objective: train PCA on benign-train, score by SPE, set the P99 threshold,
and report all six metrics on benign-test + mixed attacks.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
print("ROOT =", ROOT)

In [ ]:
import numpy as np
import pandas as pd

from src import BENIGN_LABEL, CATEGORY_COL, FLOW_ID_COL
from src.evaluation import evaluate, save_metrics_csv, write_experiment_summary
from src.pca_detector import PCADetector
from src.preprocessing import Preprocessor, clean_frame, split_one_class
from src.data_loader import load_dataset
from src.feature_selection import select_features
from src.thresholding import fit_threshold, predict, save_thresholds
from src.visualization import (plot_frontier, plot_roc_pr, plot_score_hist,
                                 plot_variance_elbow)

PROC = ROOT / "data" / "processed"
TAB = ROOT / "results" / "tables"
FIG = ROOT / "results" / "figures"
REP = ROOT / "results" / "reports"

z = np.load(PROC / "processed_A.npz", allow_pickle=True)
Xtr, Xb, Xa, yte = z["X_train"], z["X_test_benign"], z["X_test_attack"], z["y_test"]
print("loaded:", Xtr.shape, Xb.shape, Xa.shape)

In [ ]:
det = PCADetector().fit(Xtr)   # 95% variance rule, k < n (ERD C4)
det.save(PROC / "pca_model.pkl")
print(f"k={det.k_}  variance_retained={det.variance_retained_:.3f}")
plot_variance_elbow(det.variance_table(), det.k_, FIG / "variance_elbow.png")

train_spe, _ = det.score(Xtr)
SWEEP = ["p95", "p99", "p99.5", "mean+2std", "mean+3std", "mad"]
thr_map = {s: fit_threshold(train_spe, s) for s in SWEEP}
save_thresholds(PROC / "thresholds.json", thr_map)
print("P99 =", round(thr_map["p99"].value, 2))

In [ ]:
A = load_dataset(ROOT / "data" / "raw" / "synthetic_A.csv")
keptA, _ = select_features(A)
Ac, _ = clean_frame(A, keptA)
sp = split_one_class(Ac)
pp = Preprocessor.load(PROC / "scaler.pkl")
base = thr_map["p99"]

rows = []
for role, fr in (("test_benign", sp["test_benign"]), ("test_attack", sp["test_attack"])):
    spe, mse = det.score(pp.transform(fr))
    rows.append(pd.DataFrame({
        FLOW_ID_COL: fr[FLOW_ID_COL].values,
        "y_true": (fr[CATEGORY_COL] != BENIGN_LABEL).astype(int).values,
        CATEGORY_COL: fr[CATEGORY_COL].values, "split_role": role,
        "score": spe, "mse": mse}))
sdf = pd.concat(rows, ignore_index=True)
sdf["y_pred"] = predict(sdf["score"].to_numpy(), base)
sdf.to_csv(TAB / "scores_exp1.csv", index=False)

m1 = pd.DataFrame([
    {"k": det.k_, "strategy": s, "threshold": th.value,
     **evaluate(sdf["y_true"].to_numpy(), sdf["score"].to_numpy(),
                  predict(sdf["score"].to_numpy(), th))}
    for s, th in thr_map.items()])
save_metrics_csv(TAB / "metrics_exp1.csv", m1)
m1.round(4)

In [ ]:
from IPython.display import Image, display

ben_s = sdf[sdf.y_true == 0]["score"].to_numpy()
att_s = sdf[sdf.y_true == 1]["score"].to_numpy()
plot_score_hist(ben_s, att_s, base.value, FIG / "score_hist_exp1.png")
plot_roc_pr(sdf["y_true"].to_numpy(), sdf["score"].to_numpy(), FIG / "exp1")
plot_frontier(m1[["strategy", "fpr", "recall"]], FIG / "threshold_frontier_exp1.png")
display(Image(str(FIG / "score_hist_exp1.png")))

r99 = m1[m1.strategy == "p99"].iloc[0]
write_experiment_summary(
    REP / "exp1_summary.md", exp_id="Exp1", title="Basic anomaly detection",
    train_desc=f"synthetic_A benign-train (n={len(Xtr)})", test_desc=f"benign-test + mixed attacks (n={len(yte)})",
    model_desc=f"PCA k={det.k_}, variance={det.variance_retained_:.3f}",
    threshold_desc=f"P99 = {base.value:.2f}", metrics=m1.round(4),
    figures=["score_hist_exp1.png", "exp1_roc.png", "exp1_pr.png", "threshold_frontier_exp1.png"],
    worked=[f"AUROC={r99.auroc:.3f}, AUPRC={r99.auprc:.3f}"],
    failed=[f"recall@P99={r99.recall:.2f}: near-manifold attacks leak (see Exp2)"] if r99.recall < 0.8 else ["none material"],
    leakage_note="scaler + PCA + threshold fit on benign-train only (EDR V-01)",
    notebook="notebooks/03_pca_baseline.ipynb")
print("Exp1 done: AUROC=%.3f recall@P99=%.3f" % (r99.auroc, r99.recall))

## Checkpoint — Exp1 verdict

- PCA separates mixed attacks threshold-free (AUROC ≈ 0.94) but the fixed P99
  operating point misses near-manifold traffic — the subject of Exp2.